In [1]:
import pandas as pd

In [2]:
Orders = pd.read_csv('E_Commerce_Orders.csv')

In [3]:
import sqlite3
conn = sqlite3.connect(":memory:")

In [4]:
Orders.to_sql('Orders', conn, index=False, if_exists="replace")

1200

In [6]:
query = """
SELECT *
FROM Orders
"""

result = pd.read_sql(query, conn)
result

,OrderID,Date,CustomerID,Product,Quantity,UnitPrice,ShippingAddress,PaymentMethod,OrderStatus,TrackingNumber,ItemsInCart,CouponCode,ReferralSource,TotalPrice
0,ORD200000,2023-01-04,C72649,Monitor,5,570.62,928 Main St,Debit Card,Shipped,TRK37947903,7,SAVE10,Instagram,2853.10
1,ORD200001,2024-08-23,C75739,Phone,2,151.35,823 Main St,Online,Shipped,TRK91186779,3,SAVE10,Referral,302.70
2,ORD200002,2024-02-27,C81728,Tablet,5,550.68,512 Main St,Credit Card,Cancelled,TRK42903982,8,FREESHIP,Email,2753.40
3,ORD200003,2023-10-15,C33540,Chair,1,273.19,275 Main St,Debit Card,Returned,TRK62788070,5,SAVE10,Facebook,273.19
4,ORD200004,2025-05-08,C81840,Printer,4,626.01,668 Main St,Online,Delivered,TRK29241424,8,SAVE10,Email,2504.04
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1195,ORD201195,2024-06-20,C21126,Desk,1,107.04,392 Main St,Credit Card,Cancelled,TRK38009181,6,FREESHIP,Google,107.04
1196,ORD201196,2024-03-04,C20095,Monitor,2,662.53,778 Main St,Online,Cancelled,TRK69207593,5,None,Facebook,1325.06
1197,ORD201197,2023-07-13,C79674,Tablet,2,436.84,275 Main St,Online,Delivered,TRK88039356,2,FREESHIP,Instagram,873.68
1198,ORD201198,2024-08-22,C64753,Chair,4,262.52,509 Main St,Debit Card,Cancelled,TRK71683331,4,WINTER15,Instagram,1050.08


In [10]:
query2 = """
-- Profitability by payment method & coupon usage

SELECT
    CASE
        WHEN CouponCode IS NULL OR TRIM(CouponCode) = ''
        THEN 'No Coupon'
        ELSE 'Coupon Used'
    END AS coupon_status,
    PaymentMethod,
    ROUND(SUM(TotalPrice), 2) AS revenue,
    COUNT(*) AS orders,
    SUM(CASE WHEN OrderStatus = 'Cancelled' THEN 1 ELSE 0 END) AS cancelled_orders,
    ROUND(1.0 * SUM(CASE WHEN OrderStatus = 'Cancelled' THEN 1 ELSE 0 END) / COUNT(*), 2) AS cancellation_rate,
    ROUND(100.0 * SUM(TotalPrice)/ SUM(SUM(TotalPrice)) OVER (PARTITION BY PaymentMethod), 2) AS coupon_revenue_share
FROM Orders
GROUP BY PaymentMethod, coupon_status;
"""

result = pd.read_sql(query2, conn)
result 

,coupon_status,PaymentMethod,revenue,orders,cancelled_orders,cancellation_rate,coupon_revenue_share
0,Coupon Used,Cash,191109.57,183,35,0.19,73.56
1,No Coupon,Cash,68676.72,63,14,0.22,26.44
2,Coupon Used,Credit Card,209250.88,180,47,0.26,79.31
3,No Coupon,Credit Card,54596.75,54,7,0.13,20.69
4,Coupon Used,Debit Card,171188.14,169,35,0.21,73.67
5,No Coupon,Debit Card,61173.04,63,9,0.14,26.33
6,Coupon Used,Gift Card,190739.54,181,37,0.20,77.43
7,No Coupon,Gift Card,55584.38,49,13,0.27,22.57
8,Coupon Used,Online,180072.42,178,38,0.21,68.61
9,No Coupon,Online,82370.52,80,15,0.19,31.39


In [11]:
query3 = """
-- Profitability by marketing channels & coupon usage

SELECT 
    CASE
        WHEN CouponCode IS NULL OR TRIM(CouponCode) = ''
        THEN 'No Coupon'
        ELSE 'Coupon Used'
    END AS coupon_status,
    ReferralSource,
    ROUND(SUM(TotalPrice), 2) AS revenue,
    COUNT(*) AS orders,
    SUM(CASE WHEN OrderStatus = 'Cancelled' THEN 1 ELSE 0 END) AS cancelled_orders,
    ROUND(1.0 * SUM(CASE WHEN OrderStatus = 'Cancelled' THEN 1 ELSE 0 END) / COUNT(*), 2) AS cancellation_rate,
    ROUND(100.0 * SUM(TotalPrice)/ SUM(SUM(TotalPrice)) OVER (PARTITION BY ReferralSource), 2) AS coupon_revenue_share
FROM Orders
GROUP BY ReferralSource, coupon_status; 
"""

result = pd.read_sql(query3, conn)
result 

,coupon_status,ReferralSource,revenue,orders,cancelled_orders,cancellation_rate,coupon_revenue_share
0,Coupon Used,Email,195234.84,195,49,0.25,74.57
1,No Coupon,Email,66573.71,55,10,0.18,25.43
2,Coupon Used,Facebook,173951.15,161,27,0.17,69.47
3,No Coupon,Facebook,76459.75,67,15,0.22,30.53
4,Coupon Used,Google,198869.08,175,45,0.26,79.41
5,No Coupon,Google,51572.40,66,13,0.20,20.59
6,Coupon Used,Instagram,212627.74,198,35,0.18,77.24
7,No Coupon,Instagram,62657.71,61,6,0.10,22.76
8,Coupon Used,Referral,161677.74,162,36,0.22,71.28
9,No Coupon,Referral,65137.84,60,14,0.23,28.72


In [33]:
import plotly.graph_objects as go
import streamlit as st

# Define color palettes for consistency across both figures
bar_colors = {
    "Coupon Used": "#2b5c8f",  # Deep corporate blue
    "No Coupon": "#74a9cf"     # Soft light blue
}

line_colors = {
    "Coupon Used": "#2c3e50",  # Dark slate gray
    "No Coupon": "#e74c3c"     # Muted crimson red for contrast
}

# Payment Method

payment_data["PaymentMethod"] = pd.Categorical(
    payment_data["PaymentMethod"],
    categories=payment_order,
    ordered=True
)
payment_data = payment_data.sort_values("PaymentMethod")

# Create figure
payment_fig = go.Figure()

# Revenue bars
for coupon in payment_data["CouponStatus"].unique():
    subset = payment_data[payment_data["CouponStatus"] == coupon]
    payment_fig.add_trace(
        go.Bar(
            x=subset["PaymentMethod"],
            y=subset["revenue"],
            name=f"{coupon} Revenue",
            marker_color=bar_colors.get(coupon, "#333333"),
            yaxis="y1"
        )
    )

# cancelation rate line
for coupon in payment_data["CouponStatus"].unique():
    subset = payment_data[payment_data["CouponStatus"] == coupon]
    payment_fig.add_trace(
        go.Scatter(
            x=subset["PaymentMethod"],
            y=subset["cancellation_rate"],
            mode="lines+markers",
            name=f"{coupon} Cancellation Rate",
            line=dict(dash="dash", width=3, color=line_colors.get(coupon, "#333333")),
            yaxis="y2"
        )
    )

# Layout
payment_fig.update_layout(
    title="Revenue & Cancellation Rate by Payment Method and Coupon Usage",
    barmode="group",
    template="plotly_white",
    width=850,
    height=450,
    margin=dict(r=220),
    xaxis=dict(title="Payment Method", showgrid=False),
    yaxis=dict(title="Revenue"),
    yaxis2=dict(
        title="Cancellation Rate",
        overlaying="y",
        side="right",
        tickformat=".0%",
        range=[0, 1],
        showgrid=True
    ),
    legend=dict(x=1.15, y=1, xanchor="left", yanchor="top")
)

payment_fig.show()

from IPython.display import display, HTML

def notebook_insight_box(title, text):
    box_html = f"""
    <div style="
        padding: 16px;
        background-color: #f8fafc;
        border-left: 4px solid #2b5c8f;
        border-radius: 6px;
        margin: 15px 0;
        font-family: sans-serif;
        box-shadow: 0 1px 3px rgba(0,0,0,0.05);
    ">
        <span style="font-weight: bold; color: #2b5c8f; font-size: 1.1em;"> {title}</span>
        <p style="margin-top: 8px; color: #334155; line-height: 1.5; font-size: 0.95em;">{text}</p>
    </div>
    """
    display(HTML(box_html))

notebook_insight_box(
    title="Key Insight & Operational Risk",
    text="We observe differences in revenue contribution between payment methods depending on coupon usage. This may indicate that discounts affect purchasing behavior differently across payment types.<br><br>Additionally, credit card payments generate the highest revenue but also exhibit the highest cancellation rate, indicating potential post-purchase behavior issues."
)

# Referral Source

referral_data["ReferralSource"] = pd.Categorical(
    referral_data["ReferralSource"],
    categories=referral_order,
    ordered=True
)
referral_data = referral_data.sort_values("ReferralSource")

# Create figure
referral_fig = go.Figure()

# Revenue bars
for coupon in referral_data["CouponStatus"].unique():
    subset = referral_data[referral_data["CouponStatus"] == coupon]
    referral_fig.add_trace(
        go.Bar(
            x=subset["ReferralSource"],
            y=subset["revenue"],
            name=f"{coupon} Revenue",
            marker_color=bar_colors.get(coupon, "#333333"),
            yaxis="y1"
        )
    )

# cancelation rate line
for coupon in referral_data["CouponStatus"].unique():
    subset = referral_data[referral_data["CouponStatus"] == coupon]
    referral_fig.add_trace(
        go.Scatter(
            x=subset["ReferralSource"],
            y=subset["cancellation_rate"],
            mode="lines+markers",
            name=f"{coupon} Cancellation Rate",
            line=dict(dash="dash", width=3, color=line_colors.get(coupon, "#333333")),
            yaxis="y2"
        )
    )

# Layout
referral_fig.update_layout(
    title="Revenue & Cancellation Rate by Referral Source and Coupon Usage",
    barmode="group",
    template="plotly_white",
    width=850,
    height=450,
    margin=dict(r=220),
    xaxis=dict(title="Referral Source", showgrid=False),
    yaxis=dict(title="Revenue"),
    yaxis2=dict(
        title="Cancellation Rate",
        overlaying="y",
        side="right",
        tickformat=".0%",
        range=[0, 1],
        showgrid=True
    ),
    legend=dict(x=1.15, y=1, xanchor="left", yanchor="top")
)

referral_fig.show()

from IPython.display import display, HTML

def notebook_insight_box(title, text):
    box_html = f"""
    <div style="
        padding: 16px;
        background-color: #f8fafc;
        border-left: 4px solid #2b5c8f;
        border-radius: 6px;
        margin: 15px 0;
        font-family: sans-serif;
        box-shadow: 0 1px 3px rgba(0,0,0,0.05);
    ">
        <span style="font-weight: bold; color: #2b5c8f; font-size: 1.1em;"> {title}</span>
        <p style="margin-top: 8px; color: #334155; line-height: 1.5; font-size: 0.95em;">{text}</p>
    </div>
    """
    display(HTML(box_html))

notebook_insight_box(
    title="Key Insight & Operational Risk",
    text="We observe differences in revenue contribution between payment methods depending on coupon usage. This may indicate that discounts affect purchasing behavior differently across payment types.<br><br>Additionally, credit card payments generate the highest revenue but also exhibit the highest cancellation rate, indicating potential post-purchase behavior issues."
)